[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

# class MultiHeadAttention:
#     def __init__(self, d_model: int, num_heads: int):
#         self.d_model = d_model
#         self.num_heads = num_heads
#         self.d_k = d_model // num_heads

#         self.W_q = torch.nn.Linear(d_model, d_model)
#         self.W_k = torch.nn.Linear(d_model, d_model)
#         self.W_v = torch.nn.Linear(d_model, d_model)
#         self.W_o = torch.nn.Linear(d_model, d_model)

#     def forward(self, Q, K, V):
#         batch_size = Q.size(0)

#         # Linear projections
#         Q = self.W_q(Q)  # (batch_size, seq_len, d_model)
#         K = self.W_k(K)  # (batch_size, seq_len, d_model)
#         V = self.W_v(V)  # (batch_size, seq_len, d_model)

#         # Reshape for multi-head attention
#         Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)
#         K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)
#         V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)

#         # Scaled dot-product attention
#         scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch_size, num_heads, seq_len_q, seq_len_k)
#         attn_weights = torch.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)
#         attn_output = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len_q, d_k)

#         # Concatenate heads and apply final linear layer
#         attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)  # (batch_size, seq_len_q, d_model)
#         output = self.W_o(attn_output)  # (batch_size, seq_len_q, d_model)

#         return output

In [4]:
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        self.d_model = d_model
        self.num_heads = num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        # Q: (batch_size, seq_len_q, d_model)
        # K: (batch_size, seq_len_k, d_model)
        # V: (batch_size, seq_len_k, d_model)

        B = Q.size(0)

        q = self.W_q(Q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len_q, d_k)
        k = self.W_k(K).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len_k, d_k)
        v = self.W_v(V).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len_k, d_k)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)  # (batch_size, num_heads, seq_len_q, seq_len_k)
        attn_weights = torch.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)
        attn_output = attn_weights @ v  # (batch_size, num_heads, seq_len_q, d_k)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, -1, self.d_model)  # (batch_size, seq_len_q, d_model)
        output = self.W_o(attn_output)  # (batch_size, seq_len_q, d_model)
        return output

In [5]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (1.7ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.6ms)
  ✅ [3/6] Numerical correctness vs reference (1.4ms)
  ✅ [4/6] Gradient flow (19.5ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.5ms)
  ✅ [6/6] Different heads give different outputs (0.7ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (24.5ms total)
  Progress saved. Run status() to see your dashboard.

